# Passenger Fare Prediction — Linear Regression

### Machine Learning Mini Project

This notebook uses the supplied **Titanic `test(1).csv`** data to build a Linear Regression model.

**Goal:** estimate a passenger's `Fare` from a small set of numerical passenger attributes.

> This notebook is intentionally organized differently from the reference notebook while keeping the same core machine-learning workflow.


## 1. Import the required libraries

We will use Pandas for data handling, Matplotlib for visualization, and Scikit-learn for model training and evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Read the supplied dataset

Keep `test(1).csv` in the same folder as this notebook when running it in VS Code.


In [ ]:
data = pd.read_csv("test(1).csv")

print("Dataset loaded successfully")
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])

data.head()


## 3. Choose the variables for regression

For this version, the continuous target is **Fare**.  
The input variables are `Pclass`, `Age`, `SibSp`, and `Parch`.

Text columns are not used in this version, making the notebook simpler and clearly different from the reference.


In [ ]:
model_data = data[["Pclass", "Age", "SibSp", "Parch", "Fare"]].copy()

print("Selected data shape:", model_data.shape)
model_data.head()


## 4. Clean missing values

Linear Regression cannot train with missing numerical values. We replace missing `Age` and `Fare` values with their respective medians.


In [ ]:
print("Missing values before cleaning:")
print(model_data.isnull().sum())

model_data["Age"] = model_data["Age"].fillna(model_data["Age"].median())
model_data["Fare"] = model_data["Fare"].fillna(model_data["Fare"].median())

print("\nMissing values after cleaning:")
print(model_data.isnull().sum())


## 5. Prepare X and y

`X` contains the independent variables and `y` contains the value the model will predict.


In [ ]:
X = model_data[["Pclass", "Age", "SibSp", "Parch"]]
y = model_data["Fare"]

print("Input features:")
print(X.columns.tolist())

print("\nTarget variable:")
print("Fare")


## 6. Create training and testing groups

80% of the records are used for training and 20% are kept for testing.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))


## 7. Train the Linear Regression model

The model learns a linear relationship between the passenger features and fare.


In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)

print("Model training completed.")
print("Intercept:", round(regressor.intercept_, 4))


## 8. Inspect the learned coefficients

A coefficient shows how the predicted fare changes with a feature while the other features are held constant.


In [ ]:
coefficient_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": regressor.coef_
})

coefficient_table["Coefficient"] = coefficient_table["Coefficient"].round(4)
coefficient_table


## 9. Generate predictions

The trained model is now applied to the unseen testing data.


In [ ]:
predicted_fare = regressor.predict(X_test)

comparison = pd.DataFrame({
    "Actual Fare": y_test.values,
    "Predicted Fare": predicted_fare
})

comparison["Predicted Fare"] = comparison["Predicted Fare"].round(2)
comparison.head(10)


## 10. Evaluate the model

We use MAE, MSE, RMSE and R² to measure prediction performance.


In [ ]:
mae = mean_absolute_error(y_test, predicted_fare)
mse = mean_squared_error(y_test, predicted_fare)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predicted_fare)

print("===== MODEL RESULTS =====")
print("MAE :", round(mae, 4))
print("MSE :", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))


## 11. Visualize actual and predicted fares

Points closer to the diagonal line indicate predictions closer to the actual fare.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, predicted_fare, alpha=0.65)

low = min(y_test.min(), predicted_fare.min())
high = max(y_test.max(), predicted_fare.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Actual Fare vs Predicted Fare")
plt.grid(True)
plt.show()


## 12. Residual check

A residual is the difference between the actual fare and the predicted fare.


In [ ]:
residuals = y_test.values - predicted_fare

plt.figure(figsize=(8, 5))
plt.scatter(predicted_fare, residuals, alpha=0.65)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted Fare")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.grid(True)
plt.show()


## 13. Try a new passenger

The following example estimates the fare for a passenger with Pclass = 2, Age = 30, SibSp = 1 and Parch = 0.


In [ ]:
new_passenger = pd.DataFrame({
    "Pclass": [2],
    "Age": [30],
    "SibSp": [1],
    "Parch": [0]
})

estimated_fare = regressor.predict(new_passenger)[0]

print("Estimated Fare:", round(estimated_fare, 2))


## 14. Final summary

The model was trained using **418 records** from the supplied `test(1).csv` file.

### Obtained evaluation values

- **MAE:** 26.8724
- **MSE:** 1475.9896
- **RMSE:** 38.4186
- **R²:** 0.3058

The model demonstrates a measurable linear relationship between the selected passenger attributes and `Fare`. The R² value also shows that these four features alone do not explain all fare variation.

**Important:** This version predicts the continuous `Fare` column because the supplied `test(1).csv` does not contain the `Survived` target column.
